<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/hw4_olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4 — DML, DDL та складні SQL-вирази (Olist)
**GoIT Neoversity · Relational Databases for DS · Тема 4**

Джерело: [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). Усі аналітичні результати обчислюються з реальних даних під час виконання; жодні результати не підставляються вручну. Запустіть **Runtime → Restart session and run all** і збережіть notebook з outputs.

**Гранулярність:** customer_id — запис покупця для конкретного замовлення; customer_unique_id — реальний покупець; order_item — позиція замовлення; review — відгук до замовлення, не до продавця.

## 1. Setup і завантаження даних

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 70.2 MB/s eta 0:00:00


In [2]:
import os, glob, shutil, pathlib
import pandas as pd
import kagglehub
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def query(sql, params=None, max_rows=20):
    with engine.connect() as conn:
        result = pd.read_sql_query(text(sql), conn, params=params)
    display(result.head(max_rows))
    print(f'Returned rows: {len(result):,}')
    return result

def execute(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

def returning(sql, max_rows=10):
    with engine.begin() as conn:
        res = conn.execute(text(sql))
        rows = res.fetchmany(max_rows)
        cols = res.keys()
    result = pd.DataFrame(rows, columns=cols)
    display(result)
    return result

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print('Dataset location:', dataset_dir)
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}
# KaggleHub може повертати каталог із вкладеними папками.
paths = {}
for table_name, filename in files.items():
    matches = list(pathlib.Path(dataset_dir).rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f'{filename}: found {len(matches)} matches')
    paths[table_name] = matches[0]
    print(table_name, '->', matches[0])

Dataset location: /kaggle/input/brazilian-ecommerce
olist_customers_raw -> /kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv
olist_orders_raw -> /kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv
olist_order_items_raw -> /kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv
olist_products_raw -> /kaggle/input/brazilian-ecommerce/olist_products_dataset.csv
olist_sellers_raw -> /kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv
olist_order_reviews_raw -> /kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv


In [4]:
# RAW tables: зберігаємо всі поля як TEXT, щоб не втратити початкові значення.
with engine.begin() as conn:
    for table_name in files:
        conn.execute(text(f'DROP TABLE IF EXISTS {table_name} CASCADE'))

for table_name, path in paths.items():
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    frame.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=5000)
    print(f'{table_name}: {len(frame):,} rows, {len(frame.columns)} columns')

olist_customers_raw: 99,441 rows, 5 columns
olist_orders_raw: 99,441 rows, 8 columns
olist_order_items_raw: 112,650 rows, 7 columns
olist_products_raw: 32,951 rows, 9 columns
olist_sellers_raw: 3,095 rows, 4 columns
olist_order_reviews_raw: 99,224 rows, 7 columns


### Typed-схема
У typed-таблицях використовуються первинні та зовнішні ключі, доменні CHECK, правильні числові типи та timestamps. Оригінальні назви `product_name_lenght` і `product_description_lenght` з CSV виправляються тільки в typed-схемі.

In [5]:
ddl = """
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;
CREATE TABLE olist_customers (
 customer_id TEXT PRIMARY KEY, customer_unique_id TEXT NOT NULL,
 customer_zip_code_prefix INTEGER, customer_city TEXT, customer_state CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers(customer_unique_id);
CREATE TABLE olist_orders (
 order_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
 order_status TEXT NOT NULL, order_purchase_timestamp TIMESTAMP NOT NULL,
 order_approved_at TIMESTAMP, order_delivered_carrier_date TIMESTAMP,
 order_delivered_customer_date TIMESTAMP, order_estimated_delivery_date TIMESTAMP,
 CHECK (order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled'))
);
CREATE TABLE olist_products (
 product_id TEXT PRIMARY KEY, product_category_name TEXT,
 product_name_length INTEGER CHECK(product_name_length IS NULL OR product_name_length>=0),
 product_description_length INTEGER CHECK(product_description_length IS NULL OR product_description_length>=0),
 product_photos_qty INTEGER CHECK(product_photos_qty IS NULL OR product_photos_qty>=0),
 product_weight_g INTEGER CHECK(product_weight_g IS NULL OR product_weight_g>=0),
 product_length_cm INTEGER CHECK(product_length_cm IS NULL OR product_length_cm>=0),
 product_height_cm INTEGER CHECK(product_height_cm IS NULL OR product_height_cm>=0),
 product_width_cm INTEGER CHECK(product_width_cm IS NULL OR product_width_cm>=0)
);
CREATE TABLE olist_sellers (
 seller_id TEXT PRIMARY KEY, seller_zip_code_prefix INTEGER, seller_city TEXT,
 seller_state CHAR(2) NOT NULL
);
CREATE TABLE olist_order_items (
 order_id TEXT NOT NULL REFERENCES olist_orders(order_id), order_item_id INTEGER NOT NULL,
 product_id TEXT REFERENCES olist_products(product_id), seller_id TEXT REFERENCES olist_sellers(seller_id),
 shipping_limit_date TIMESTAMP,
 price NUMERIC(12,2) CHECK(price IS NULL OR price>=0),
 freight_value NUMERIC(12,2) CHECK(freight_value IS NULL OR freight_value>=0),
 PRIMARY KEY(order_id,order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items(seller_id,order_id);
CREATE TABLE olist_order_reviews (
 review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
 review_id TEXT, order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
 review_score SMALLINT CHECK(review_score BETWEEN 1 AND 5),
 review_comment_title TEXT, review_comment_message TEXT,
 review_creation_date TIMESTAMP, review_answer_timestamp TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews(order_id);
"""
with engine.begin() as conn:
    for statement in ddl.split(';'):
        if statement.strip(): conn.execute(text(statement))
print('Typed schema created')

Typed schema created


In [6]:
inserts = [
"""INSERT INTO olist_customers SELECT customer_id, customer_unique_id,
 NULLIF(customer_zip_code_prefix,'')::INTEGER, customer_city, customer_state::CHAR(2)
 FROM olist_customers_raw""",
"""INSERT INTO olist_orders SELECT order_id, customer_id, order_status,
 NULLIF(order_purchase_timestamp,'')::TIMESTAMP, NULLIF(order_approved_at,'')::TIMESTAMP,
 NULLIF(order_delivered_carrier_date,'')::TIMESTAMP,
 NULLIF(order_delivered_customer_date,'')::TIMESTAMP,
 NULLIF(order_estimated_delivery_date,'')::TIMESTAMP FROM olist_orders_raw""",
"""INSERT INTO olist_products SELECT product_id, NULLIF(product_category_name,''),
 NULLIF(product_name_lenght,'')::INTEGER, NULLIF(product_description_lenght,'')::INTEGER,
 NULLIF(product_photos_qty,'')::INTEGER, NULLIF(product_weight_g,'')::INTEGER,
 NULLIF(product_length_cm,'')::INTEGER, NULLIF(product_height_cm,'')::INTEGER,
 NULLIF(product_width_cm,'')::INTEGER FROM olist_products_raw""",
"""INSERT INTO olist_sellers SELECT seller_id, NULLIF(seller_zip_code_prefix,'')::INTEGER,
 seller_city, seller_state::CHAR(2) FROM olist_sellers_raw""",
"""INSERT INTO olist_order_items SELECT order_id, order_item_id::INTEGER,
 product_id, seller_id, NULLIF(shipping_limit_date,'')::TIMESTAMP,
 NULLIF(price,'')::NUMERIC(12,2), NULLIF(freight_value,'')::NUMERIC(12,2)
 FROM olist_order_items_raw""",
"""INSERT INTO olist_order_reviews(review_id,order_id,review_score,review_comment_title,
 review_comment_message,review_creation_date,review_answer_timestamp)
 SELECT review_id,order_id,NULLIF(review_score,'')::SMALLINT,
 NULLIF(review_comment_title,''),NULLIF(review_comment_message,''),
 NULLIF(review_creation_date,'')::TIMESTAMP,NULLIF(review_answer_timestamp,'')::TIMESTAMP
 FROM olist_order_reviews_raw"""
]
with engine.begin() as conn:
    for sql in inserts: conn.execute(text(sql))
print('Typed inserts completed')

Typed inserts completed


In [7]:
counts = query("""
SELECT 'customers' table_name, (SELECT COUNT(*) FROM olist_customers_raw) raw_rows, COUNT(*) typed_rows FROM olist_customers
UNION ALL SELECT 'orders',(SELECT COUNT(*) FROM olist_orders_raw),COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items',(SELECT COUNT(*) FROM olist_order_items_raw),COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products',(SELECT COUNT(*) FROM olist_products_raw),COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers',(SELECT COUNT(*) FROM olist_sellers_raw),COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews',(SELECT COUNT(*) FROM olist_order_reviews_raw),COUNT(*) FROM olist_order_reviews
ORDER BY table_name
""")
assert (counts.raw_rows == counts.typed_rows).all(), 'Raw/typed row count mismatch'
print('Interpretation: усі записи з raw перенесено без втрат; правила FK/CHECK не відкинули рядки.')

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


Returned rows: 6
Interpretation: усі записи з raw перенесено без втрат; правила FK/CHECK не відкинули рядки.


## 2. DML — INSERT, UPSERT, UPDATE, DELETE

### 2.1 INSERT ... RETURNING
**Business-question:** як автоматично зафіксувати продавця з найбільшою кількістю запізнілих доставок?

`RETURNING` одразу показує створений запис для аудиту. Для повторного запуску вставка перевіряє відсутність аналогічного alert, щоб не створювати дублікати.

In [8]:
execute("""CREATE TABLE seller_alerts (
 alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
 seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
 alert_type TEXT NOT NULL CHECK(alert_type IN ('late_delivery','low_rating','data_quality')),
 severity SMALLINT NOT NULL CHECK(severity BETWEEN 1 AND 5),
 details TEXT, created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
)""")
alert_sql = """
WITH seller_orders AS (
 SELECT DISTINCT seller_id, order_id FROM olist_order_items WHERE seller_id IS NOT NULL
), late_sellers AS (
 SELECT so.seller_id, COUNT(DISTINCT o.order_id) late_orders
 FROM seller_orders so JOIN olist_orders o ON o.order_id=so.order_id
 WHERE o.order_delivered_customer_date IS NOT NULL
 AND o.order_estimated_delivery_date IS NOT NULL
 AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
 GROUP BY so.seller_id ORDER BY late_orders DESC, so.seller_id LIMIT 1
)
INSERT INTO seller_alerts(seller_id,alert_type,severity,details)
SELECT ls.seller_id,'late_delivery',4,
 'Seller has '||ls.late_orders||' late delivered orders in the loaded dataset'
FROM late_sellers ls
WHERE NOT EXISTS (SELECT 1 FROM seller_alerts a
 WHERE a.seller_id=ls.seller_id AND a.alert_type='late_delivery')
RETURNING alert_id,seller_id,alert_type,severity,created_at
"""
alert_result = returning(alert_sql)
print('Audit: inserted alerts:', len(alert_result))

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-10-27 13:08:53.570130+00:00


Audit: inserted alerts: 1


### 2.2 ON CONFLICT DO UPDATE — seller_score
**Business-question:** як отримати оцінку кожного продавця без множення відгуків через order items?

`SELECT DISTINCT seller_id, order_id` фіксує правильну гранулярність. Відгуки попередньо агрегуються до `order_id`, потім розраховується середня оцінка продавця. `ON CONFLICT` оновлює наявний seller замість дублювання. Відсутність відгуків означає `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`. Відгук стосується всього замовлення; у multi-seller orders він приписується кожному продавцю — це обмеження даних.

In [9]:
execute("""CREATE TABLE seller_score (
 seller_id TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
 avg_rating NUMERIC(3,2), n_reviews INTEGER NOT NULL DEFAULT 0 CHECK(n_reviews>=0),
 tier TEXT CHECK(tier IN ('gold','silver','bronze')),
 updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
)""")
score_sql = """
WITH seller_orders AS (
 SELECT DISTINCT seller_id,order_id FROM olist_order_items WHERE seller_id IS NOT NULL
), review_per_order AS (
 SELECT order_id, AVG(review_score)::NUMERIC(3,2) order_review_score
 FROM olist_order_reviews WHERE review_score IS NOT NULL GROUP BY order_id
), seller_rating AS (
 SELECT so.seller_id, AVG(rpo.order_review_score)::NUMERIC(3,2) avg_rating,
 COUNT(rpo.order_id)::INTEGER n_reviews
 FROM seller_orders so LEFT JOIN review_per_order rpo ON rpo.order_id=so.order_id
 GROUP BY so.seller_id
)
INSERT INTO seller_score(seller_id,avg_rating,n_reviews,tier)
SELECT seller_id,avg_rating,n_reviews,
 CASE WHEN avg_rating IS NULL THEN NULL WHEN avg_rating>=4.5 THEN 'gold'
 WHEN avg_rating>=3.5 THEN 'silver' ELSE 'bronze' END
FROM seller_rating
ON CONFLICT(seller_id) DO UPDATE SET avg_rating=EXCLUDED.avg_rating,
 n_reviews=EXCLUDED.n_reviews,tier=EXCLUDED.tier,updated_at=NOW()
RETURNING seller_id,avg_rating,n_reviews,tier,updated_at
"""
score_result = returning(score_sql)
score_counts = query("SELECT tier,COUNT(*) n_sellers FROM seller_score GROUP BY tier ORDER BY n_sellers DESC")
print('Interpretation: seller_score заповнено, повторний upsert оновить записи за seller_id.')

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-27 13:08:53.763355+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-27 13:08:53.763355+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-27 13:08:53.763355+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-27 13:08:53.763355+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-27 13:08:53.763355+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-27 13:08:53.763355+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-27 13:08:53.763355+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-27 13:08:53.763355+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-27 13:08:53.763355+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-27 13:08:53.763355+00:00


,tier,n_sellers
0,silver,1568
1,gold,983
2,bronze,539
3,None,5


Returned rows: 4
Interpretation: seller_score заповнено, повторний upsert оновить записи за seller_id.


### 2.3 UPDATE ... FROM
**Business-question:** які замовлення були доставлені пізніше обіцяної дати?

Прапорець обчислюється з актуальних дат, тому повторне виконання дає той самий результат. NULL-дати не позначаються як запізнення.

In [10]:
execute("ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE")
update_sql = """
WITH delivery_flags AS (
 SELECT order_id,(order_delivered_customer_date IS NOT NULL
 AND order_estimated_delivery_date IS NOT NULL
 AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) is_late_calc
 FROM olist_orders
)
UPDATE olist_orders o SET is_late=f.is_late_calc
FROM delivery_flags f WHERE f.order_id=o.order_id
RETURNING o.order_id,o.customer_id,o.is_late
"""
# Не виводимо всі ~100 тис. рядків у notebook.
with engine.begin() as conn:
    updated = conn.execute(text(update_sql))
    updated_sample = pd.DataFrame(updated.fetchmany(10),columns=updated.keys())
display(updated_sample)
late_stats = query('SELECT is_late, COUNT(*) n_orders FROM olist_orders GROUP BY is_late ORDER BY is_late')

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


,is_late,n_orders
0,False,92906
1,True,6535


Returned rows: 2


### 2.4 DELETE ... RETURNING
**Business-question:** які alert-записи старші за 30 днів потрібно прибрати?

На новому датасеті таблиця alert створена щойно, тому видалень може бути 0. Це коректний результат.

In [11]:
deleted_alerts = returning("""
DELETE FROM seller_alerts WHERE created_at < NOW()-INTERVAL '30 days'
RETURNING alert_id,seller_id,alert_type,created_at
""")
print('Interpretation: removed outdated alerts:',len(deleted_alerts))

,alert_id,seller_id,alert_type,created_at


Interpretation: removed outdated alerts: 0


## 3. Customer segmentation — RFM-like features

**Business-question:** як сегментувати реальних покупців, враховуючи повторні покупки?

`customer_unique_id` відображає реального покупця, на відміну від `customer_id` для конкретного замовлення. `olist_customer_dim` дає один рядок на покупця для зовнішнього ключа. Reference date береться як день після останнього замовлення в датасеті, а не поточна дата — інакше історична recency залежала б від дня запуску. `ON CONFLICT` робить сегментацію повторюваною.

In [12]:
execute("""CREATE TABLE olist_customer_dim (
 customer_unique_id TEXT PRIMARY KEY,
 latest_customer_id TEXT REFERENCES olist_customers(customer_id),
 customer_state CHAR(2), customer_city TEXT
)""")
execute("""INSERT INTO olist_customer_dim(customer_unique_id,latest_customer_id,customer_state,customer_city)
SELECT DISTINCT ON(c.customer_unique_id) c.customer_unique_id,c.customer_id,c.customer_state,c.customer_city
FROM olist_customers c LEFT JOIN olist_orders o ON o.customer_id=c.customer_id
ORDER BY c.customer_unique_id,o.order_purchase_timestamp DESC NULLS LAST,c.customer_id""")
execute("""CREATE TABLE customer_segments (
 segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
 customer_unique_id TEXT NOT NULL UNIQUE REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE,
 segment_name TEXT NOT NULL CHECK(segment_name IN ('VIP','regular','new','inactive')),
 rfm_score SMALLINT NOT NULL CHECK(rfm_score BETWEEN 1 AND 5),
 recency_days INTEGER CHECK(recency_days IS NULL OR recency_days>=0),
 monetary_value NUMERIC(12,2) NOT NULL DEFAULT 0 CHECK(monetary_value>=0),
 n_orders INTEGER NOT NULL DEFAULT 0 CHECK(n_orders>=0),
 assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
)""")

In [13]:
segments_sql = """
WITH reference_date AS (
 SELECT (MAX(order_purchase_timestamp)::DATE+1) as_of_date FROM olist_orders
), order_totals AS (
 SELECT o.order_id,o.customer_id,o.order_purchase_timestamp::DATE order_date,
 COALESCE(SUM(oi.price+oi.freight_value),0)::NUMERIC(12,2) order_total
 FROM olist_orders o LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id
 GROUP BY o.order_id,o.customer_id,o.order_purchase_timestamp
), customer_stats AS (
 SELECT c.customer_unique_id,COUNT(DISTINCT ot.order_id)::INTEGER n_orders,
 COALESCE(SUM(ot.order_total),0)::NUMERIC(12,2) monetary_value,
 MAX(ot.order_date) last_order_date
 FROM olist_customer_dim cd JOIN olist_customers c ON c.customer_unique_id=cd.customer_unique_id
 LEFT JOIN order_totals ot ON ot.customer_id=c.customer_id GROUP BY c.customer_unique_id
), segmented AS (
 SELECT cs.customer_unique_id,cs.n_orders,cs.monetary_value,
 CASE WHEN cs.last_order_date IS NULL THEN NULL
 ELSE (rd.as_of_date-cs.last_order_date)::INTEGER END recency_days,
 CASE WHEN cs.monetary_value>=1000 THEN 'VIP'
 WHEN cs.last_order_date IS NOT NULL AND (rd.as_of_date-cs.last_order_date)::INTEGER<=90
 AND cs.n_orders=1 THEN 'new'
 WHEN cs.n_orders>=2 OR cs.monetary_value>=100 THEN 'regular' ELSE 'inactive' END segment_name,
 CASE WHEN cs.monetary_value>=1000 THEN 5 WHEN cs.monetary_value>=500 THEN 4
 WHEN cs.n_orders>=2 THEN 3 WHEN cs.n_orders=1 THEN 2 ELSE 1 END::SMALLINT rfm_score
 FROM customer_stats cs CROSS JOIN reference_date rd
)
INSERT INTO customer_segments(customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders)
SELECT customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders FROM segmented
ON CONFLICT(customer_unique_id) DO UPDATE SET
 segment_name=EXCLUDED.segment_name,rfm_score=EXCLUDED.rfm_score,
 recency_days=EXCLUDED.recency_days,monetary_value=EXCLUDED.monetary_value,
 n_orders=EXCLUDED.n_orders,assigned_at=NOW()
RETURNING customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
"""
with engine.begin() as conn:
    res=conn.execute(text(segments_sql))
    sample=pd.DataFrame(res.fetchmany(10),columns=res.keys())
display(sample)
segments_summary=query('SELECT segment_name,COUNT(*) n_customers,ROUND(AVG(monetary_value),2) avg_monetary FROM customer_segments GROUP BY segment_name ORDER BY n_customers DESC')
constraints=query("SELECT conname,contype,pg_get_constraintdef(oid) definition FROM pg_constraint WHERE conrelid='customer_segments'::regclass ORDER BY conname")

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


,segment_name,n_customers,avg_monetary
0,regular,45982,224.57
1,inactive,40032,59.28
2,new,8891,140.39
3,VIP,1191,1591.81


Returned rows: 4


,conname,contype,definition
0,customer_segments_customer_unique_id_fkey,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...
1,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
2,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
3,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
4,customer_segments_pkey,p,PRIMARY KEY (segment_id)
5,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
6,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
7,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."


Returned rows: 8


## 4. JOIN — п’ять бізнес-запитів

### 4.1 INNER JOIN (4 таблиці)
**Business-question:** Які категорії товарів та штати продавців мають найбільший revenue для delivered orders?

In [14]:
result_25 = query("""SELECT p.product_category_name,s.seller_state,COUNT(DISTINCT oi.order_id) n_orders,
SUM(oi.price+oi.freight_value)::NUMERIC(12,2) revenue
FROM olist_order_items oi JOIN olist_orders o ON o.order_id=oi.order_id
JOIN olist_products p ON p.product_id=oi.product_id
JOIN olist_sellers s ON s.seller_id=oi.seller_id
WHERE o.order_status='delivered'
GROUP BY p.product_category_name,s.seller_state ORDER BY revenue DESC NULLS LAST LIMIT 10""")
print('Interpretation: найбільший revenue:', result_25.iloc[0].to_dict() if len(result_25) else 'no rows')

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


Returned rows: 10
Interpretation: найбільший revenue: {'product_category_name': 'cama_mesa_banho', 'seller_state': 'SP', 'n_orders': 8214, 'revenue': 1070048.14}


### 4.2 LEFT JOIN + COALESCE
**Business-question:** Які customer-level features можна обчислити без дублювання review score?

In [15]:
result_27 = query("""WITH order_totals AS (
SELECT o.order_id,c.customer_unique_id,COALESCE(SUM(oi.price+oi.freight_value),0)::NUMERIC(12,2) order_total
FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id
LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id GROUP BY o.order_id,c.customer_unique_id
), review_per_order AS (
SELECT order_id,AVG(review_score)::NUMERIC(3,2) order_review_score
FROM olist_order_reviews WHERE review_score IS NOT NULL GROUP BY order_id
), customer_stats AS (
SELECT ot.customer_unique_id,COUNT(DISTINCT ot.order_id) n_orders,
COALESCE(SUM(ot.order_total),0)::NUMERIC(12,2) total_spend,
AVG(rpo.order_review_score)::NUMERIC(3,2) avg_review_score
FROM order_totals ot LEFT JOIN review_per_order rpo ON rpo.order_id=ot.order_id
GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id,cd.customer_state,COALESCE(cs.n_orders,0) n_orders,
COALESCE(cs.total_spend,0)::NUMERIC(12,2) total_spend,
COALESCE(cs.avg_review_score,0)::NUMERIC(3,2) avg_review_score
FROM olist_customer_dim cd LEFT JOIN customer_stats cs ON cs.customer_unique_id=cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST LIMIT 20""")
print('Interpretation: найбільші витрати покупця:', result_27.iloc[0].to_dict() if len(result_27) else 'no rows'); print('Review агреговано до order_id перед JOIN, отже позиції не множать оцінку.')

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


Returned rows: 20
Interpretation: найбільші витрати покупця: {'customer_unique_id': '0a0a92112bd4c708ca5fde585afaa872', 'customer_state': 'RJ', 'n_orders': 1, 'total_spend': 13664.08, 'avg_review_score': 1.0}
Review агреговано до order_id перед JOIN, отже позиції не множать оцінку.


### 4.3 FULL OUTER JOIN
**Business-question:** Які штати представлені лише покупцями, лише продавцями або обома?

In [16]:
result_29 = query("""WITH customer_states AS (SELECT customer_state state,COUNT(*) n_customers FROM olist_customers GROUP BY customer_state),
seller_states AS (SELECT seller_state state,COUNT(*) n_sellers FROM olist_sellers GROUP BY seller_state)
SELECT COALESCE(c.state,s.state) state,c.n_customers,s.n_sellers,
CASE WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
WHEN c.state IS NOT NULL THEN 'customers_only' ELSE 'sellers_only' END state_presence
FROM customer_states c FULL OUTER JOIN seller_states s ON s.state=c.state ORDER BY state""")
print('Interpretation: розподіл штатів:',result_29.state_presence.value_counts().to_dict())

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


Returned rows: 27
Interpretation: розподіл штатів: {'both': 23, 'customers_only': 4}


### 4.4 SELF JOIN
**Business-question:** Які пари продавців працюють у тому самому штаті, але в різних містах?

In [17]:
result_31 = query("""SELECT s1.seller_id seller_a,s2.seller_id seller_b,s1.seller_state,
s1.seller_city city_a,s2.seller_city city_b
FROM olist_sellers s1 JOIN olist_sellers s2
ON s1.seller_state=s2.seller_state AND s1.seller_id<s2.seller_id
AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state,seller_a,seller_b LIMIT 20""")
print('Interpretation: показано',len(result_31),'прикладів пар; умова seller_a < seller_b прибирає дзеркальні дублікати.')

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


Returned rows: 20
Interpretation: показано 20 прикладів пар; умова seller_a < seller_b прибирає дзеркальні дублікати.


### 4.5 Anti-join
**Business-question:** Які замовлення не мають жодного review?

In [18]:
result_33 = query("""SELECT o.order_id,o.customer_id,o.order_status,o.order_purchase_timestamp
FROM olist_orders o LEFT JOIN olist_order_reviews rv ON rv.order_id=o.order_id
WHERE rv.review_row_id IS NULL ORDER BY o.order_purchase_timestamp LIMIT 20""")
print('Interpretation: прикладів замовлень без відгуків:',len(result_33),'; LIMIT 20 не дорівнює загальній кількості.')

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


Returned rows: 20
Interpretation: прикладів замовлень без відгуків: 20 ; LIMIT 20 не дорівнює загальній кількості.


## 5. GROUP BY, HAVING, STRING_AGG

### 5.1 Revenue per category
**Business-question:** Які категорії мають gross revenue понад 10 000 BRL?

In [19]:
result_36 = query("""SELECT p.product_category_name,COUNT(DISTINCT oi.order_id) n_orders,
SUM(oi.price)::NUMERIC(12,2) gross_revenue,
SUM(oi.freight_value)::NUMERIC(12,2) freight,
AVG(oi.price)::NUMERIC(8,2) avg_item_price
FROM olist_order_items oi JOIN olist_products p ON p.product_id=oi.product_id
GROUP BY p.product_category_name HAVING SUM(oi.price)>10000
ORDER BY gross_revenue DESC NULLS LAST""")
print('Interpretation: категорій вище порогу:',len(result_36),'; лідер:',result_36.iloc[0].to_dict() if len(result_36) else 'none')

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


Returned rows: 53
Interpretation: категорій вище порогу: 53 ; лідер: {'product_category_name': 'beleza_saude', 'n_orders': 8836, 'gross_revenue': 1258681.34, 'freight': 182566.73, 'avg_item_price': 130.16}


### 5.2 AOV per state
**Business-question:** У яких штатах із принаймні 100 замовленнями найвищий AOV?

In [20]:
result_38 = query("""WITH order_totals AS (
SELECT o.order_id,c.customer_state,SUM(oi.price+oi.freight_value)::NUMERIC(12,2) order_total
FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id
JOIN olist_order_items oi ON oi.order_id=o.order_id GROUP BY o.order_id,c.customer_state)
SELECT customer_state,COUNT(*) n_orders,SUM(order_total)::NUMERIC(12,2) gmv,
(SUM(order_total)/NULLIF(COUNT(*),0))::NUMERIC(8,2) aov
FROM order_totals GROUP BY customer_state HAVING COUNT(*)>=100
ORDER BY aov DESC NULLS LAST""")
print('Interpretation: штат із найбільшим AOV:',result_38.iloc[0].to_dict() if len(result_38) else 'none')

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


Returned rows: 24
Interpretation: штат із найбільшим AOV: {'customer_state': 'PB', 'n_orders': 532, 'gmv': 140987.81, 'aov': 265.01}


### 5.3 STRING_AGG
**Business-question:** Які товарні категорії купували покупці з найбільшими витратами?

In [21]:
result_40 = query("""SELECT c.customer_unique_id,COUNT(DISTINCT o.order_id) n_orders,
SUM(oi.price)::NUMERIC(12,2) total_spend,
STRING_AGG(DISTINCT COALESCE(p.product_category_name,'(unknown)'),', '
ORDER BY COALESCE(p.product_category_name,'(unknown)')) categories
FROM olist_customers c JOIN olist_orders o ON o.customer_id=c.customer_id
JOIN olist_order_items oi ON oi.order_id=o.order_id
LEFT JOIN olist_products p ON p.product_id=oi.product_id
GROUP BY c.customer_unique_id HAVING COUNT(DISTINCT o.order_id)>=1
ORDER BY total_spend DESC NULLS LAST LIMIT 20""")
print('Interpretation: найбільші витрати:',result_40.iloc[0].to_dict() if len(result_40) else 'none')

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


Returned rows: 20
Interpretation: найбільші витрати: {'customer_unique_id': '0a0a92112bd4c708ca5fde585afaa872', 'n_orders': 1, 'total_spend': 13440.0, 'categories': 'telefonia_fixa'}


**WHERE vs HAVING:** `WHERE` відбирає окремі рядки до `GROUP BY` (наприклад, лише delivered orders); `HAVING` фільтрує вже агреговані групи (наприклад, категорії із сумою продажів понад 10 000 або штати зі щонайменше 100 замовленнями).

## 6. Set operation, window function і data-quality

### 6.1 INTERSECT
**Business-question:** У яких штатах є і клієнти, і продавці?

In [22]:
result_44 = query("""SELECT customer_state AS state FROM olist_customers
INTERSECT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state""")
print('Interpretation: штатів, спільних для обох множин:',len(result_44)); print('INTERSECT залишає лише спільні унікальні state.')

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


Returned rows: 23
Interpretation: штатів, спільних для обох множин: 23
INTERSECT залишає лише спільні унікальні state.


### 6.2 ROW_NUMBER()
**Business-question:** Яке останнє замовлення кожного реального покупця?

In [23]:
result_46 = query("""WITH ranked_orders AS (
SELECT c.customer_unique_id,o.customer_id,o.order_id,o.order_purchase_timestamp,
ROW_NUMBER() OVER(PARTITION BY c.customer_unique_id
ORDER BY o.order_purchase_timestamp DESC,o.order_id DESC) rn
FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id)
SELECT customer_unique_id,customer_id,order_id,order_purchase_timestamp
FROM ranked_orders WHERE rn=1 ORDER BY customer_unique_id LIMIT 20""")
print('Interpretation: один останній order на customer_unique_id, із детермінованим tie-break за order_id.')

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


Returned rows: 20
Interpretation: один останній order на customer_unique_id, із детермінованим tie-break за order_id.


In [24]:
quality=query("""
SELECT
 (SELECT COUNT(*) FROM olist_orders o WHERE NOT EXISTS
 (SELECT 1 FROM olist_order_reviews r WHERE r.order_id=o.order_id)) orders_without_reviews,
 (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL OR product_category_name='') products_without_category,
 (SELECT COUNT(*) FROM olist_order_reviews WHERE review_comment_message IS NULL OR review_comment_message='') reviews_without_message,
 (SELECT COUNT(*) FROM olist_orders WHERE order_status='canceled') canceled_orders,
 (SELECT COUNT(*) FROM olist_customers)- (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) repeated_customer_ids
""")
print('Interpretation: відсутність review або тексту коментаря не обов’язково означає помилку; для ML це окремі missingness indicators.')

,orders_without_reviews,products_without_category,reviews_without_message,canceled_orders,repeated_customer_ids
0,768,610,58247,625,3345


Returned rows: 1
Interpretation: відсутність review або тексту коментаря не обов’язково означає помилку; для ML це окремі missingness indicators.


## Reflection (200–250 слів)

Найскладнішим для мене був FULL OUTER JOIN, тому що на відміну від INNER JOIN він зберігає записи з обох таблиць навіть без відповідної пари. У порівнянні штатів покупців і продавців важливо правильно обробити NULL після об’єднання та відокремити категорії «тільки покупці», «тільки продавці» й «обидва». Найскладніше бізнес-питання стосувалося оцінки продавців: відгук в Olist прив’язаний до замовлення, а не до конкретної позиції чи продавця.

Найкориснішою PostgreSQL-можливістю для production ETL вважаю ON CONFLICT DO UPDATE. Вона дозволяє повторно обчислювати метрики й оновлювати таблиці без створення дублікатів. RETURNING доповнює цей механізм, надаючи результати операцій для аудиту. У реальному ML-проєкті window functions застосувала б для пошуку останньої покупки, розрахунку інтервалів між замовленнями та створення часових ознак без порушення послідовності подій.

Аудит Olist показує, чому відсутній відгук, порожній коментар або невідома категорія потребують окремої інтерпретації. Такі значення не слід автоматично видаляти: вони можуть відображати нормальну поведінку користувачів або обмеження збору даних. Для customer-level ML необхідно використовувати customer_unique_id, оскільки customer_id часто відображає лише конкретне замовлення; інакше повторні покупки будуть розділені між різними записами. Перед обчисленням seller_score дані потрібно привести до гранулярності seller_id × order_id, а review агрегувати до order_id. Інакше JOIN із позиціями замовлення штучно збільшить вагу окремих оцінок і спотворить рейтинг. Для надійного pipeline також потрібні перевірки ключів, контроль дублювань, документація гранулярності та моніторинг пропусків.

## Фінальний контроль

In [25]:
assert len(counts)==6 and (counts.raw_rows==counts.typed_rows).all()
assert len(constraints)>=5
with engine.connect() as conn:
    n_customers=conn.execute(text('SELECT COUNT(*) FROM olist_customer_dim')).scalar_one()
    n_segments=conn.execute(text('SELECT COUNT(*) FROM customer_segments')).scalar_one()
    n_sellers=conn.execute(text('SELECT COUNT(*) FROM seller_score')).scalar_one()
    assert n_customers==n_segments
    assert n_sellers>0
    print('customer_dim:',n_customers,'segments:',n_segments,'seller_score:',n_sellers)
print('FINAL CHECK PASSED')

customer_dim: 96096 segments: 96096 seller_score: 3095
FINAL CHECK PASSED
